In [1]:
# pip install tensorflow pandas scikit-learn


In [2]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense


2025-08-19 16:05:30.417127: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-19 16:05:30.457246: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-08-19 16:05:31.520247: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:

# -----------------------------
# 0) Data: use existing df or sample
# -----------------------------
try:
    assert 'df' in globals()
    assert {'text','label'}.issubset(df.columns)
    print("Using provided DataFrame `df`.")
except:
    print("No `df` found — using a small sample dataset.")
    df = pd.DataFrame({
        "text": [
            "I love this movie",
            "Worst film ever",
            "Not bad could be better",
            "Absolutely fantastic",
            "Terrible acting and boring",
            "I really enjoyed it",
        ],
        "label": [1, 0, 1, 1, 0, 1]
    })


No `df` found — using a small sample dataset.


In [4]:

# -----------------------------
# 1) Clean text (simple)
# -----------------------------
def clean_text(s):
    s = s.lower()
    s = re.sub(r"[^a-z\s']", " ", s) # keep letters/apostrophes
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text"] = df["text"].astype(str).apply(clean_text)
df["label"] = df["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.33, random_state=42, stratify=df["label"]
)


In [5]:

# -----------------------------
# 2) Tokenize + PAD (many-to-one requires fixed length for batching)
# -----------------------------
MAX_VOCAB = 10000 # keep top words
MAX_LEN = 20 # pad / truncate to this length

tok = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tok.fit_on_texts(X_train)

def to_padded(texts):
    seqs = tok.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding="post", truncating="post", value=0)

Xtr = to_padded(X_train)
Xte = to_padded(X_test)
ytr = y_train.values
yte = y_test.values

print("\nSample sequences BEFORE padding:", tok.texts_to_sequences(X_train[:2]))
print("Sample sequences AFTER padding:\n", Xtr[:2])
print("Each row length:", Xtr.shape[1], "(= MAX_LEN)")



Sample sequences BEFORE padding: [[2, 3, 4, 5, 6], [7, 8, 9, 10]]
Sample sequences AFTER padding:
 [[ 2  3  4  5  6  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 7  8  9 10  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]]
Each row length: 20 (= MAX_LEN)


In [6]:

# -----------------------------
# 3) Simple RNN (many-to-one)
# Embedding: converts token IDs to vectors
# SimpleRNN: returns FINAL hidden state (sequence summary)
# -----------------------------
EMBED_DIM = 32
RNN_UNITS = 32

model = Sequential([
    Embedding(
        input_dim=min(MAX_VOCAB, len(tok.word_index) + 1),
        output_dim=EMBED_DIM,
        input_length=MAX_LEN,
        mask_zero=True # <-- tells RNN to ignore padding (zeros)
    ),
    SimpleRNN(RNN_UNITS), # final hidden state only -> many-to-one
    Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
print("\nModel summary:")
model.summary()



Model summary:


/home/akashs/.local/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
E0000 00:00:1755599732.017931   15171 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1755599732.023077   15171 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [7]:

# -----------------------------
# 4) Train
# -----------------------------
history = model.fit(
    Xtr, ytr,
    epochs=8,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# -----------------------------
# 5) Evaluate
# -----------------------------
loss, acc = model.evaluate(Xte, yte, verbose=0)
print(f"\nTest accuracy: {acc:.3f}")

Epoch 1/8
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.3333 - loss: 0.7188 - val_accuracy: 0.0000e+00 - val_loss: 0.6953
Epoch 2/8
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.6667 - loss: 0.6937 - val_accuracy: 0.0000e+00 - val_loss: 0.6934
Epoch 3/8
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.6667 - loss: 0.6696 - val_accuracy: 1.0000 - val_loss: 0.6916
Epoch 4/8
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.6667 - loss: 0.6463 - val_accuracy: 1.0000 - val_loss: 0.6898
Epoch 5/8
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.6667 - loss: 0.6235 - val_accuracy: 1.0000 - val_loss: 0.6880
Epoch 6/8
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 1.0000 - loss: 0.6013 - val_accuracy: 1.0000 - val_loss: 0.6864
Epoch 7/8
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 1.0000 - loss: 0.5794 - val_accuracy: 1.0000 - val_loss: 0.6848
Epoch 8/8
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 1.0000 - loss: 0.5577 - val_accuracy: 1.0000 - val_loss: 0.6833

T

In [9]:

# -----------------------------
# 6) Predict on new sentences
# -----------------------------
def predict_sentiment(texts):
    if isinstance(texts, str):
        texts = [texts]
    cleaned = [clean_text(t) for t in texts]
    pad = to_padded(cleaned)
    probs = model.predict(pad, verbose=0).ravel()
    labels = (probs >= 0.5).astype(int)
    return list(zip(texts, probs, ["positive" if i==1 else "negative" for i in labels]))

examples = [
    "I absolutely loved it!",
    "This was terrible and boring",
    "Not bad… could be better",
    "Average movie, nothing special",
    "Good movie",
    "Bad acting"
]
for txt, p, lab in predict_sentiment(examples):
    print(f"{lab:9s} | {p:.3f} | {txt}")

positive  | 0.524 | I absolutely loved it!
negative  | 0.495 | This was terrible and boring
positive  | 0.630 | Not bad… could be better
positive  | 0.510 | Average movie, nothing special
positive  | 0.538 | Good movie
negative  | 0.498 | Bad acting
